# Quantum Learning Procedure

File này là bước đầu tiên trong pipeline, chịu trách nhiệm:
1. Load toàn bộ datasets (synthetic + real-world)
2. Chạy quantum kernel evaluation trên tất cả combinations: 9 ansätze × 3 ML models × 10 runs
3. Tính 24 complexity metrics cho mỗi dataset
4. Tạo Synthetic Training Dataset (features + labels) cho bước Recommendation tiếp theo

Output: 3 file CSV trong `results/` — taska_labels, taskb_labels, và synthesis_df.

In [3]:
from Qsun.Qkernels import *
from Qsun.Qgates import *
from Qsun.Qmeas import *
from Qsun.Qcircuit import *
from Qsun.Qwave import *
from Qsun.Qencodes import *
from Qsun.Qdata import *

import numpy as np
import matplotlib.pyplot as plt
#from datasets.benchmark_datasets import *
from datasets.load_data import *
from src.kernel_evaluation import *
from src.config import *

import problexity as px

np.random.seed(1234)

Load hàm `load_extended_datasets` từ `datasets/load_data` với `max_qubit=4`, giới hạn số features bằng PCA để tương thích với quantum kernel computation (4 features ↔ 4 qubits).

In [4]:
datasets = load_extended_datasets(max_qubit=4, verbose=True)
synthetic_prefixes = ['Blobs', 'Circle', 'Moons', 'XOR', 'Spiral', 'Checkerboard']
real_prefixes = ['Iris', 'Wine', 'Breast', 'Pima', 'Banknote', 'Haberman']

for prefix in synthetic_prefixes + real_prefixes:
    count = sum(1 for k in datasets if k.startswith(prefix))
    print(f"  {prefix}: {count}")

LOADING DATASETS
  Blobs Easy      : 24 loaded, 0 skipped
  Blobs Hard      : 18 loaded, 0 skipped
  Circles         : 24 loaded, 0 skipped
  Moons           : 21 loaded, 0 skipped
  Concentric Rings: 24 loaded, 0 skipped
  XOR             : 18 loaded, 0 skipped
  Spiral          : 18 loaded, 0 skipped
  Checkerboard    : 27 loaded, 0 skipped

  Real-World      : 6 loaded, 0 skipped
  Subsamples      : 20 loaded, 1 skipped

Total: 200 datasets (synthetic: 174, real-world: 26)
  Blobs: 42
  Circle: 24
  Moons: 21
  XOR: 18
  Spiral: 18
  Checkerboard: 27
  Iris: 2
  Wine: 4
  Breast: 5
  Pima: 5
  Banknote: 5
  Haberman: 5


Kiểm tra số lượng datasets theo từng prefix. Synthetic datasets có nhiều variants (khác noise, samples, parameters) để tăng tổng số training samples. Real-world datasets lấy từ các benchmark chuẩn (UCI, OpenML).

In [5]:
datasets = load_extended_datasets(max_qubit=4, verbose=True)
for name, (X_tr, X_te, y_tr, y_te) in datasets.items():
    print(f"\n{name} dataset: Train {X_tr.shape}, Test {X_te.shape}")

LOADING DATASETS
  Blobs Easy      : 24 loaded, 0 skipped
  Blobs Hard      : 18 loaded, 0 skipped
  Circles         : 24 loaded, 0 skipped
  Moons           : 21 loaded, 0 skipped
  Concentric Rings: 24 loaded, 0 skipped
  XOR             : 18 loaded, 0 skipped
  Spiral          : 18 loaded, 0 skipped
  Checkerboard    : 27 loaded, 0 skipped

  Real-World      : 6 loaded, 0 skipped
  Subsamples      : 20 loaded, 1 skipped

Total: 200 datasets (synthetic: 174, real-world: 26)

Blobs_F2C2_S100 dataset: Train (80, 2), Test (20, 2)

Blobs_F2C2_S150 dataset: Train (120, 2), Test (30, 2)

Blobs_F2C2_S200 dataset: Train (160, 2), Test (40, 2)

Blobs_F2C3_S100 dataset: Train (80, 2), Test (20, 2)

Blobs_F2C3_S150 dataset: Train (120, 2), Test (30, 2)

Blobs_F2C3_S200 dataset: Train (160, 2), Test (40, 2)

Blobs_F2C4_S100 dataset: Train (80, 2), Test (20, 2)

Blobs_F2C4_S150 dataset: Train (120, 2), Test (30, 2)

Blobs_F2C4_S200 dataset: Train (160, 2), Test (40, 2)

Blobs_F3C2_S100 dataset: T

### Model Execution

`total_runs`: hàm chính chạy quantum kernel evaluation cho 1 dataset. Quy trình mỗi run:
- Tính kernel matrix cho tất cả 9 encodings bằng `total_kernels`
- Evaluate mỗi kernel matrix với 3 ML models (SVM, GPC, KRC) qua `evaluate_kernel`
- Tích lũy train/test accuracy qua n_runs, sau đó tính mean ± std

`run_all_datasets`: wrapper chạy `total_runs` trên toàn bộ datasets.

In [6]:
def total_runs(dataset_name="Iris",
               datasets=None,
               encodings=None,
               n_layers=2,
               n_runs=10,
               test_size=0.2,
               random_state=42,
               include_variants=True):

    if encodings is None:
        encodings = get_available_encodings()

    print(f"Dataset: {dataset_name}")

    results_accumulator = {
        enc: {
            m: {"train": [], "test": []}
            for m in ["SVM", "GPC", "KRC"]
        }
        for enc in encodings
    }

    for run in range(n_runs):
        seed = random_state + run

        X_train, X_test, y_train, y_test = datasets[dataset_name]

        kernels = total_kernels(
            X_train, X_test,
            encoding_names=encodings,
            n_layers=n_layers,
            random_state=seed,
            verbose=False
        )

        for enc_name, (K_train, K_test) in kernels.items():
            for model_name in ["SVM", "GPC", "KRC"]:
                result = evaluate_kernel(
                    K_train, K_test, y_train, y_test,
                    enc_name, model_name
                )
                results_accumulator[enc_name][model_name]["train"].append(
                    result.train_accuracy
                )
                results_accumulator[enc_name][model_name]["test"].append(
                    result.test_accuracy
                )

        print(f"  Run {run+1}/{n_runs} completed", end='\r')

    print()

    all_results = {}
    for enc_name in encodings:
        enc_results = []
        for model_name in ["SVM", "GPC", "KRC"]:
            train_scores = results_accumulator[enc_name][model_name]["train"]
            test_scores = results_accumulator[enc_name][model_name]["test"]

            enc_results.append(KernelEvaluation(
                model_name=model_name,
                encoding_name=enc_name,
                train_accuracy=np.mean(train_scores),
                test_accuracy=np.mean(test_scores),
                train_std=np.std(train_scores),
                test_std=np.std(test_scores)
            ))
        all_results[enc_name] = enc_results

    return {"results": all_results}


def run_all_datasets(encodings=None, 
                     n_layers=2, 
                     n_runs=10, 
                     test_size=0.2, 
                     random_state=42, 
                     include_variants=True):

    datasets = load_extended_datasets(max_qubit=4, verbose=True)
   
    all_results = {}
    dataset_names = list(datasets.keys())
    
    print(f"Total datasets to process: {len(dataset_names)}")
    print("=" * 60)
    
    for idx, dataset_name in enumerate(dataset_names, 1):
        print(f"\n[{idx}/{len(dataset_names)}] Processing {dataset_name}...")
        
        result = total_runs(
            dataset_name=dataset_name,
            encodings=encodings,
            n_layers=n_layers,
            n_runs=n_runs,
            test_size=test_size,
            random_state=random_state,
            include_variants=include_variants
        )
        
        all_results[dataset_name] = result
    
    print("\n" + "=" * 60)
    print("All datasets processed!")
    
    return all_results

### Summary & Visualization Utilities

- `summary`: in bảng accuracy cho từng encoding × model combination của 1 dataset
- `create_summary_table`: tạo DataFrame tổng hợp accuracy của 1 model cụ thể (vd: SVM) trên tất cả datasets × encodings — dùng cho bảng so sánh trong paper

In [7]:
def summary(all_results):
    print(f"{'Encoding':<22} {'Model':<6} {'Train':<18} {'Test':<18}")
    print("-" * 75)
    
    best_test_acc = 0
    best_config = None
    
    for encoding_name, results in all_results.items():
        for r in results: 
            train_str = f"{r.train_accuracy:.4f} ± {r.train_std:.4f}"
            test_str = f"{r.test_accuracy:.4f} ± {r.test_std:.4f}"
            print(f"{r.encoding_name:<22} {r.model_name:<6} {train_str:<18} {test_str:<18}")
            
            if r.test_accuracy > best_test_acc:
                best_test_acc = r.test_accuracy
                best_config = r
    
    print("-" * 75)
    if best_config:
        print(f"Best: {best_config.encoding_name} + {best_config.model_name} "
              f"= {best_test_acc:.4f} ± {best_config.test_std:.4f}")

In [8]:
def create_summary_table(all_dataset_results, 
                         model_name="SVM",
                         show_std=False) -> pd.DataFrame:
    encodings = list(ENCODING_REGISTER.keys())
    
    table_data = []
    for dataset_name, result_dict in all_dataset_results.items():
        row = {"Dataset": dataset_name}
        results = result_dict["results"]
        
        for enc_name in encodings:
            if enc_name in results:
                for r in results[enc_name]:
                    if r.model_name == model_name:
                        if show_std:
                            row[enc_name] = f"{r.test_accuracy:.4f} ± {r.test_std:.4f}"
                        else:
                            row[enc_name] = r.test_accuracy
                        break
            else:
                row[enc_name] = None if not show_std else "-"
        
        table_data.append(row)
    
    df = pd.DataFrame(table_data)
    df = df.set_index("Dataset")
    
    return df

### Main Execution

Chạy toàn bộ pipeline: load datasets → tính kernel matrices → evaluate với 3 models × 10 runs. Kết quả lưu vào `all_results` (dict: dataset_name → results dict) và serialize ra pickle file để tái sử dụng mà không cần chạy lại (bước này tốn nhiều giờ).

In [9]:
datasets = load_extended_datasets(max_qubit=4, verbose=True)

print(f"Total datasets: {len(datasets)}")
print("=" * 60)

all_results = {}

for dataset_name in datasets.keys():
    result = total_runs(
        dataset_name=dataset_name,
        datasets=datasets,         
        n_runs=10,
        random_state=42,
    )
    all_results[dataset_name] = result

print(f"\n Completely Loading datasets")

LOADING DATASETS
  Blobs Easy      : 24 loaded, 0 skipped
  Blobs Hard      : 18 loaded, 0 skipped
  Circles         : 24 loaded, 0 skipped
  Moons           : 21 loaded, 0 skipped
  Concentric Rings: 24 loaded, 0 skipped
  XOR             : 18 loaded, 0 skipped
  Spiral          : 18 loaded, 0 skipped
  Checkerboard    : 27 loaded, 0 skipped

  Real-World      : 6 loaded, 0 skipped
  Subsamples      : 20 loaded, 1 skipped

Total: 200 datasets (synthetic: 174, real-world: 26)
Total datasets: 200
Dataset: Blobs_F2C2_S100
  Run 10/10 completed
Dataset: Blobs_F2C2_S150
  Run 10/10 completed
Dataset: Blobs_F2C2_S200
  Run 10/10 completed
Dataset: Blobs_F2C3_S100
  Run 10/10 completed
Dataset: Blobs_F2C3_S150
  Run 10/10 completed
Dataset: Blobs_F2C3_S200
  Run 10/10 completed
Dataset: Blobs_F2C4_S100
  Run 10/10 completed
Dataset: Blobs_F2C4_S150
  Run 10/10 completed
Dataset: Blobs_F2C4_S200
  Run 10/10 completed
Dataset: Blobs_F3C2_S100
  Run 10/10 completed
Dataset: Blobs_F3C2_S150
  Ru

In [10]:
import pickle

with open('all_results with load_data.pkl', 'wb') as f:
    pickle.dump(all_results, f)
print(f"Saved {len(all_results)} results to all_results with load_data.pkl")

Saved 200 results to all_results with load_data.pkl


### Result Summaries

In 3 bảng tổng hợp:
1. **Best Kernel per Model**: với mỗi ML model, kernel nào tốt nhất cho từng dataset
2. **Best Model + Kernel per Dataset**: combination tốt nhất tổng thể cho từng dataset
3. **Model Comparison (Side-by-side)**: so sánh 3 models song song trên cùng 1 dataset

In [11]:
print("\n" + "="*80)
print("SUMMARY: BEST KERNEL FOR EACH MODEL (ACROSS ALL DATASETS)")
print("="*80)

for model_name in ["SVM", "GPC", "KRC"]:
    print(f"\n{'='*70}")
    print(f"MODEL: {model_name}")
    print(f"{'='*70}")
    print(f"{'Dataset':<25} {'Best Kernel':<22} {'Test Accuracy':<18}")
    print("-" * 70)
    
    for dataset_name, result_dict in all_results.items():
        results = result_dict["results"]
        
        best_acc = 0
        best_kernel = None
        best_std = 0
        
        for enc_name, enc_results in results.items():
            for r in enc_results:
                if r.model_name == model_name and r.test_accuracy > best_acc:
                    best_acc = r.test_accuracy
                    best_kernel = enc_name
                    best_std = r.test_std
        
        acc_str = f"{best_acc:.4f} ± {best_std:.4f}"
        print(f"{dataset_name:<25} {best_kernel:<22} {acc_str:<18}")
    
    print("-" * 70)


SUMMARY: BEST KERNEL FOR EACH MODEL (ACROSS ALL DATASETS)

MODEL: SVM
Dataset                   Best Kernel            Test Accuracy     
----------------------------------------------------------------------
Blobs_F2C2_S100           HighDim                1.0000 ± 0.0000   
Blobs_F2C2_S150           HighDim                1.0000 ± 0.0000   
Blobs_F2C2_S200           HighDim                1.0000 ± 0.0000   
Blobs_F2C3_S100           HighDim                1.0000 ± 0.0000   
Blobs_F2C3_S150           HighDim                1.0000 ± 0.0000   
Blobs_F2C3_S200           HighDim                1.0000 ± 0.0000   
Blobs_F2C4_S100           HighDim                1.0000 ± 0.0000   
Blobs_F2C4_S150           HighDim                1.0000 ± 0.0000   
Blobs_F2C4_S200           HighDim                1.0000 ± 0.0000   
Blobs_F3C2_S100           HZY_CZ                 1.0000 ± 0.0000   
Blobs_F3C2_S150           HighDim                1.0000 ± 0.0000   
Blobs_F3C2_S200           HighDim         

In [12]:
print("\n" + "="*80)
print("SUMMARY: BEST MODEL + KERNEL FOR EACH DATASET")
print("="*80)
print(f"{'Dataset':<25} {'Best Model':<8} {'Best Kernel':<22} {'Test Accuracy':<18}")
print("-" * 80)

for dataset_name, result_dict in all_results.items():
    results = result_dict["results"]
    
    best_acc = 0
    best_kernel = None
    best_model = None
    best_std = 0
    
    for enc_name, enc_results in results.items():
        for r in enc_results:
            if r.test_accuracy > best_acc:
                best_acc = r.test_accuracy
                best_kernel = enc_name
                best_model = r.model_name
                best_std = r.test_std
    
    acc_str = f"{best_acc:.4f} ± {best_std:.4f}"
    print(f"{dataset_name:<25} {best_model:<8} {best_kernel:<22} {acc_str:<18}")

print("-" * 80)


SUMMARY: BEST MODEL + KERNEL FOR EACH DATASET
Dataset                   Best Model Best Kernel            Test Accuracy     
--------------------------------------------------------------------------------
Blobs_F2C2_S100           SVM      HighDim                1.0000 ± 0.0000   
Blobs_F2C2_S150           SVM      HighDim                1.0000 ± 0.0000   
Blobs_F2C2_S200           SVM      HighDim                1.0000 ± 0.0000   
Blobs_F2C3_S100           SVM      HighDim                1.0000 ± 0.0000   
Blobs_F2C3_S150           SVM      HighDim                1.0000 ± 0.0000   
Blobs_F2C3_S200           SVM      HighDim                1.0000 ± 0.0000   
Blobs_F2C4_S100           SVM      HighDim                1.0000 ± 0.0000   
Blobs_F2C4_S150           SVM      HighDim                1.0000 ± 0.0000   
Blobs_F2C4_S200           SVM      HighDim                1.0000 ± 0.0000   
Blobs_F3C2_S100           SVM      HZY_CZ                 1.0000 ± 0.0000   
Blobs_F3C2_S150        

In [13]:
print("\n" + "="*80)
print("SUMMARY: MODEL COMPARISON (SIDE-BY-SIDE)")
print("="*80)

print(f"{'Dataset':<20} ", end="")
for model in ["SVM", "GPC", "KRC"]:
    print(f"{model:<25} ", end="")
print()
print("-" * 95)

for dataset_name, result_dict in all_results.items():
    results = result_dict["results"]
    print(f"{dataset_name:<20} ", end="")
    
    for model_name in ["SVM", "GPC", "KRC"]:
        best_acc = 0
        best_kernel = None
        best_std = 0
        
        for enc_name, enc_results in results.items():
            for r in enc_results:
                if r.model_name == model_name and r.test_accuracy > best_acc:
                    best_acc = r.test_accuracy
                    best_kernel = enc_name
                    best_std = r.test_std
        
        acc_str = f"{best_acc:.4f}±{best_std:.4f}"
        print(f"{acc_str:<25} ", end="")
    
    print()


SUMMARY: MODEL COMPARISON (SIDE-BY-SIDE)
Dataset              SVM                       GPC                       KRC                       
-----------------------------------------------------------------------------------------------
Blobs_F2C2_S100      1.0000±0.0000             1.0000±0.0000             1.0000±0.0000             
Blobs_F2C2_S150      1.0000±0.0000             1.0000±0.0000             1.0000±0.0000             
Blobs_F2C2_S200      1.0000±0.0000             1.0000±0.0000             1.0000±0.0000             
Blobs_F2C3_S100      1.0000±0.0000             1.0000±0.0000             1.0000±0.0000             
Blobs_F2C3_S150      1.0000±0.0000             1.0000±0.0000             1.0000±0.0000             
Blobs_F2C3_S200      1.0000±0.0000             1.0000±0.0000             1.0000±0.0000             
Blobs_F2C4_S100      1.0000±0.0000             1.0000±0.0000             1.0000±0.0000             
Blobs_F2C4_S150      1.0000±0.0000             1.0000±0.0000  

### Label Generation: Task-A & Task-B

`label_dataframe_best_model_per_dataset` tạo labels cho cả hai task formulations:

- **Task-A**: mỗi dataset → 1 label duy nhất = kernel đạt accuracy cao nhất (across best model)
- **Task-B**: mỗi dataset → nhiều labels = tất cả kernels có accuracy trong ngưỡng tolerance (threshold=0.01) so với best kernel, **chỉ xét trong best model** (không cross-model)

Lưu ý: Task-B chỉ xét tied kernels trong cùng 1 model (best model) chứ không so sánh cross-model. Ví dụ nếu SVM là best model cho dataset X, chỉ các kernels chạy trên SVM mới được xét tied.

In [14]:
def label_dataframe_best_model_per_dataset(all_dataset_results, threshold=0.01):

    encodings = list(ENCODING_REGISTER.keys())
    models = ["SVM", "GPC", "KRC"]
    
    table_data = []
    tied_data = []
    
    for dataset_name, result_dict in all_dataset_results.items():
        results = result_dict["results"]
        
        best_acc = 0
        best_kernel = None
        best_model = None
        
        for enc_name in encodings:
            if enc_name in results:
                for r in results[enc_name]:
                    if r.test_accuracy > best_acc:
                        best_acc = r.test_accuracy
                        best_kernel = enc_name
                        best_model = r.model_name
        
        if best_acc == 0:
            continue
        
        # ========== TASK-A: Single Best ==========
        table_data.append({
            "Dataset": dataset_name,
            "Best_Model": best_model,
            "Best_Kernel": best_kernel,
            "Accuracy": best_acc
        })
        
        # ========== TASK-B: Tied Kernels (CHỈ TRONG BEST MODEL) ==========
        if threshold is not None:
            # CHỈ xét các kernels trong best_model
            for enc_name in encodings:
                if enc_name in results:
                    for r in results[enc_name]:
                        # CHỈ LẤY RESULTS TỪ BEST MODEL
                        if r.model_name == best_model and r.test_accuracy >= best_acc - threshold:
                            tied_data.append({
                                "Dataset": dataset_name,
                                "Best_Model": best_model,  
                                "Best_Kernel": enc_name,
                                "Accuracy": r.test_accuracy
                            })
    
    df_best = pd.DataFrame(table_data).set_index("Dataset")
    
    if threshold is not None and tied_data:
        df_tied = pd.DataFrame(tied_data).set_index("Dataset")
        
        print("Model Distribution:")
        print("  Task 1-A (Best only):")
        model_counts = df_best["Best_Model"].value_counts()
        for model, count in model_counts.items():
            print(f"    {model}: {count} datasets ({count/len(df_best)*100:.1f}%)")
        
        print("\n  Task 1-B (Tied within best model):")
        model_counts_tied = df_tied["Best_Model"].value_counts()
        for model, count in model_counts_tied.items():
            print(f"    {model}: {count} samples ({count/len(df_tied)*100:.1f}%)")
        
        tied_per_dataset = df_tied.groupby(level=0).size()
        print(f"\n  Average tied kernels per dataset: {tied_per_dataset.mean():.2f}")
        print(f"  Min: {tied_per_dataset.min()}, Max: {tied_per_dataset.max()}")
        print()
        
        return df_best, df_tied
    
    return df_best

df_best, df_tied = label_dataframe_best_model_per_dataset(all_results, threshold=0.01)

print("="*70)
print("Task 1-A: Single BEST (Model + Kernel) per Dataset")
print("="*70)
print(df_best.head(10))
print(f"\nTotal samples: {len(df_best)}")

print("\n" + "="*70)
print("Task 1-B: Tied Kernels (within best model only)")
print("="*70)
print(df_tied.head(15))
print(f"\nTotal samples: {len(df_tied)}")

Model Distribution:
  Task 1-A (Best only):
    SVM: 152 datasets (76.0%)
    GPC: 24 datasets (12.0%)
    KRC: 24 datasets (12.0%)

  Task 1-B (Tied within best model):
    SVM: 421 samples (87.2%)
    GPC: 37 samples (7.7%)
    KRC: 25 samples (5.2%)

  Average tied kernels per dataset: 2.42
  Min: 1, Max: 6

Task 1-A: Single BEST (Model + Kernel) per Dataset


                Best_Model Best_Kernel  Accuracy
Dataset                                         
Blobs_F2C2_S100        SVM     HighDim       1.0
Blobs_F2C2_S150        SVM     HighDim       1.0
Blobs_F2C2_S200        SVM     HighDim       1.0
Blobs_F2C3_S100        SVM     HighDim       1.0
Blobs_F2C3_S150        SVM     HighDim       1.0
Blobs_F2C3_S200        SVM     HighDim       1.0
Blobs_F2C4_S100        SVM     HighDim       1.0
Blobs_F2C4_S150        SVM     HighDim       1.0
Blobs_F2C4_S200        SVM     HighDim       1.0
Blobs_F3C2_S100        SVM      HZY_CZ       1.0

Total samples: 200

Task 1-B: Tied Kernels (within best model only)
                Best_Model          Best_Kernel  Accuracy
Dataset                                                  
Blobs_F2C2_S100        SVM              HighDim       1.0
Blobs_F2C2_S100        SVM               HZY_CZ       1.0
Blobs_F2C2_S100        SVM          SeparableRX       1.0
Blobs_F2C2_S100        SVM  HardwareEfficientRx      

# Data Complexity Procedure

### Classical Metrics

Tính 24 complexity metrics cho mỗi dataset:
- 22 metrics từ Problexity (covering feature-based, linearity, neighborhood, network, dimensionality, class imbalance)
- 2 metrics từ Qsun: Kolmogorov complexity và intrinsic dimensionality

Mỗi dataset được chạy 10 lần qua Problexity rồi lấy trung bình để giảm variance. Input là tập train (X_tr) đã scale về [0, 1] — tránh data leakage từ test set.

In [15]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [16]:
datasets = load_extended_datasets(max_qubit=4, verbose=True)

complexities_datasets = {}  

for name, (X_tr, X_te, y_tr, y_te) in datasets.items():  
    complexities_train = [] 
    for i in range(10):
        cc = px.ComplexityCalculator()
        cc.fit(X_tr, y_tr)
        results = list(cc.complexity)
        results.append(kolmogorov_complex(X_tr)['best_bytes'])
        results.append(intrinsic_dim_from_cov(X_tr))
        complexities_train.append(results)

    complexities_train = np.array(complexities_train)
    complexities_datasets[name] = np.mean(complexities_train, axis=0)

LOADING DATASETS
  Blobs Easy      : 24 loaded, 0 skipped
  Blobs Hard      : 18 loaded, 0 skipped
  Circles         : 24 loaded, 0 skipped
  Moons           : 21 loaded, 0 skipped
  Concentric Rings: 24 loaded, 0 skipped
  XOR             : 18 loaded, 0 skipped
  Spiral          : 18 loaded, 0 skipped
  Checkerboard    : 27 loaded, 0 skipped

  Real-World      : 6 loaded, 0 skipped
  Subsamples      : 20 loaded, 1 skipped

Total: 200 datasets (synthetic: 174, real-world: 26)


In [17]:
labels = cc._metrics() + ['kolmogorov', 'intrinsic']
print(f"Number of metrics: {len(labels)}")
print(f"Labels: {labels}")

df = pd.DataFrame.from_dict(complexities_datasets, orient='index', columns=labels)
df

Number of metrics: 24
Labels: ['f1', 'f1v', 'f2', 'f3', 'f4', 'l1', 'l2', 'l3', 'n1', 'n2', 'n3', 'n4', 't1', 'lsc', 'density', 'clsCoef', 'hubs', 't2', 't3', 't4', 'c1', 'c2', 'kolmogorov', 'intrinsic']


,f1,f1v,f2,f3,f4,l1,l2,l3,n1,n2,...,density,clsCoef,hubs,t2,t3,t4,c1,c2,kolmogorov,intrinsic
Blobs_F2C2_S100,0.006808,0.001974,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.006250,0.016113,...,0.508228,0.003610,0.501207,0.025000,0.012500,0.5,0.000000,0.000000,0.986111,1.016451
Blobs_F2C2_S150,0.008471,0.002344,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.004167,0.014433,...,0.510224,0.009877,0.501577,0.016667,0.008333,0.5,0.000000,0.000000,0.974703,1.018584
Blobs_F2C2_S200,0.008790,0.002384,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.003125,0.012816,...,0.509434,0.010341,0.501195,0.012500,0.006250,0.5,0.000000,0.000000,0.963523,1.018072
Blobs_F2C3_S100,0.004638,0.001454,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.009376,0.017360,...,0.511342,0.003483,0.494153,0.037503,0.018751,0.5,0.000171,0.000474,0.989198,1.690700
Blobs_F2C3_S150,0.004209,0.001313,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.006250,0.013687,...,0.510865,0.007072,0.500999,0.025000,0.012500,0.5,0.000000,0.000000,0.973154,1.689884
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Haberman_S150,0.753499,0.489870,0.542453,0.858333,0.85,0.156478,0.233333,0.235000,0.150000,0.341501,...,0.915406,0.535931,0.738163,0.025000,0.025000,1.0,0.163359,0.357664,0.291336,2.608393
Wine_S80,0.162382,0.062927,0.060430,0.156250,0.00,0.003608,0.031250,0.006250,0.039062,0.292734,...,0.931052,0.564079,0.804299,0.062500,0.062500,1.0,0.006349,0.017425,0.962694,3.521621
Wine_S100,0.169052,0.050124,0.055212,0.187500,0.00,0.002746,0.012500,0.008750,0.012500,0.265798,...,0.921519,0.537679,0.783092,0.050000,0.050000,1.0,0.007226,0.019802,0.953416,3.187640
Wine_S120,0.176416,0.062447,0.044377,0.187500,0.00,0.004258,0.031250,0.015625,0.026042,0.278372,...,0.899561,0.456920,0.775964,0.041667,0.041667,1.0,0.007841,0.021468,0.950777,3.508040


Ghép label `Best_Kernel` từ Task-A vào DataFrame complexity metrics → tạo thành **Synthetic Training Dataset** hoàn chỉnh, trong đó mỗi row = 1 dataset, columns = 24 metrics + 1 label.

In [18]:
df["Best_Kernel"] = df_best["Best_Kernel"]
df

,f1,f1v,f2,f3,f4,l1,l2,l3,n1,n2,...,clsCoef,hubs,t2,t3,t4,c1,c2,kolmogorov,intrinsic,Best_Kernel
Blobs_F2C2_S100,0.006808,0.001974,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.006250,0.016113,...,0.003610,0.501207,0.025000,0.012500,0.5,0.000000,0.000000,0.986111,1.016451,HighDim
Blobs_F2C2_S150,0.008471,0.002344,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.004167,0.014433,...,0.009877,0.501577,0.016667,0.008333,0.5,0.000000,0.000000,0.974703,1.018584,HighDim
Blobs_F2C2_S200,0.008790,0.002384,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.003125,0.012816,...,0.010341,0.501195,0.012500,0.006250,0.5,0.000000,0.000000,0.963523,1.018072,HighDim
Blobs_F2C3_S100,0.004638,0.001454,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.009376,0.017360,...,0.003483,0.494153,0.037503,0.018751,0.5,0.000171,0.000474,0.989198,1.690700,HighDim
Blobs_F2C3_S150,0.004209,0.001313,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.006250,0.013687,...,0.007072,0.500999,0.025000,0.012500,0.5,0.000000,0.000000,0.973154,1.689884,HighDim
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Haberman_S150,0.753499,0.489870,0.542453,0.858333,0.85,0.156478,0.233333,0.235000,0.150000,0.341501,...,0.535931,0.738163,0.025000,0.025000,1.0,0.163359,0.357664,0.291336,2.608393,ParamZFeatureMap
Wine_S80,0.162382,0.062927,0.060430,0.156250,0.00,0.003608,0.031250,0.006250,0.039062,0.292734,...,0.564079,0.804299,0.062500,0.062500,1.0,0.006349,0.017425,0.962694,3.521621,HighDim
Wine_S100,0.169052,0.050124,0.055212,0.187500,0.00,0.002746,0.012500,0.008750,0.012500,0.265798,...,0.537679,0.783092,0.050000,0.050000,1.0,0.007226,0.019802,0.953416,3.187640,HighDim
Wine_S120,0.176416,0.062447,0.044377,0.187500,0.00,0.004258,0.031250,0.015625,0.026042,0.278372,...,0.456920,0.775964,0.041667,0.041667,1.0,0.007841,0.021468,0.950777,3.508040,SeparableRX


### Save Results

Lưu 3 file CSV:
- `taska_labels`: mỗi dataset → 1 best kernel (Task-A labels)
- `taskb_labels`: mỗi dataset → nhiều tied kernels (Task-B labels, expanded)
- `synthesis_df`: 24 complexity metrics + best kernel label (Synthetic Training Dataset)

Các file này là input cho bước tiếp theo: Majority Voting và LOOCV recommendation procedures.

In [19]:
df_best.to_csv("results/taska_labels with load_data.csv", index=True)
print("Saved: results/taska_labels with load_data.csv")

df_tied.to_csv("results/taskb_labels with load_data.csv", index=True)
print("Saved: results/taskb_labels with load_data.csv")

df.to_csv("results/synthesis_df with load_data.csv", index=True)
print("Saved: results/synthesis_df with load_data.csv")

Saved: results/taska_labels with load_data.csv
Saved: results/taskb_labels with load_data.csv
Saved: results/synthesis_df with load_data.csv
